In [5]:
from PreRun import PreRun
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error as mse
from itertools import product
from datetime import date

In [6]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [7]:
params = {
    'objective': 'regression',
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metrics': 'mse',
    'shrinkage_rate': 0.1
}

In [8]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [19]:
start_stop_dict = {
    system_id: {
        name[0]: [0, 0] for name in name_val
    } for system_id in systems_good_timezones_manual_edit
}
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        start_stop_dict[system_id].pop(name)
        continue
    if val == 'None':
        real_val = None
    else:
        real_val = val
    prerun_system = PreRun(system_id, f'./test_results/{system_id}-{val}/', real_val, systems_cleaned)
    start_stop_dict[system_id][name][0] = prerun_system.data.at[0, 'time'].date()
    last_index = prerun_system.data.index[-1]
    start_stop_dict[system_id][name][1] = prerun_system.data.at[last_index, 'time'].date()

                     time    energy
0     2007-09-01 06:00:00  0.032494
1     2007-09-01 07:00:00  0.159633
2     2007-09-01 08:00:00  0.304443
3     2007-09-01 09:00:00  0.265836
4     2007-09-01 10:00:00  0.804245
...                   ...       ...
43271 2023-02-28 13:00:00  0.610406
43272 2023-02-28 14:00:00  0.408742
43273 2023-02-28 15:00:00  0.395145
43274 2023-02-28 16:00:00  0.230329
43275 2023-02-28 17:00:00  0.003725

[43276 rows x 2 columns]
                     time    energy
0     2006-01-25 08:00:00  0.105199
1     2006-01-25 09:00:00  0.355960
2     2006-01-25 10:00:00  0.371972
3     2006-01-25 11:00:00  0.692585
4     2006-01-25 12:00:00  0.850932
...                   ...       ...
49108 2023-02-28 13:00:00  0.658749
49109 2023-02-28 14:00:00  0.446517
49110 2023-02-28 15:00:00  0.431092
49111 2023-02-28 16:00:00  0.269969
49112 2023-02-28 17:00:00  0.010433

[49113 rows x 2 columns]
                     time    energy
0     2010-11-10 06:00:00  0.001348
1     2010-1

In [20]:
start_stop_dict

{4: {'other': [datetime.date(2007, 9, 1), datetime.date(2023, 2, 28)]},
 10: {'other': [datetime.date(2006, 1, 25), datetime.date(2023, 2, 28)]},
 33: {'other': [datetime.date(2010, 11, 10), datetime.date(2023, 2, 28)]},
 36: {'other': [datetime.date(2012, 3, 30), datetime.date(2019, 7, 21)]},
 50: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 51: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 1199: {'inverter': [datetime.date(2010, 5, 29), datetime.date(2018, 8, 3)]},
 1204: {'inverter': [datetime.date(2011, 2, 9), datetime.date(2015, 1, 5)]},
 1283: {'inverter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)],
  'meter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)]},
 1284: {'other': [datetime.date(2012, 6, 30), datetime.date(2015, 3, 15)]},
 1289: {'other': [datetime.date(2012, 9, 28), datetime.date(2020, 5, 13)]},
 1332: {'inverter': [datetime.date(2013, 3, 30), datetime.date(2014, 7, 31)],
  'meter': [datetime.date(

In [14]:
def basic_lightbgm_test(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, params: dict):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=1, include_last_year=True, todays_lags=1, include_hour_cyclic=True,include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    df = prerun_system.amended_data
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '1_days_ago', '1_hours_ago_today', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_starter_train = df[df['time'] < pd.Timestamp(year=2018, month=1, day=1, hour=0)]
    X_starter_train = df_starter_train[my_cols]
    y_starter_train = df_starter_train['energy']
    lgb_tr = lgb.Dataset(X_starter_train, label=y_starter_train)
    df_starter_val = df[(df['time'] >= pd.Timestamp(year=2019, month=1, day=1, hour=0))
                        & (df['time'] < pd.Timestamp(year=2021, month=1, day=1, hour=0))]
    X_val = df_starter_val[my_cols]
    y_val = df_starter_val['energy']
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_tr)
    df_starter_test = df[df['time'] >= pd.Timestamp(year=2022, month=1, day=1, hour=0)]
    X_test = df_starter_test[my_cols]
    y_test = df_starter_test['energy']
    num_round = 100
    bst = lgb.train(params, lgb_tr, num_round, valid_sets=[lgb_val,])
    y_pred = bst.predict(X_test)
    print(f'Mean squared error: {mse(y_test, y_pred):.5f}')
    

In [15]:
basic_lightbgm_test(4, './test_results/4-None', None, systems_cleaned, params)

                     time    energy
0     2007-09-01 06:00:00  0.032494
1     2007-09-01 07:00:00  0.159633
2     2007-09-01 08:00:00  0.304443
3     2007-09-01 09:00:00  0.265836
4     2007-09-01 10:00:00  0.804245
...                   ...       ...
43271 2023-02-28 13:00:00  0.610406
43272 2023-02-28 14:00:00  0.408742
43273 2023-02-28 15:00:00  0.395145
43274 2023-02-28 16:00:00  0.230329
43275 2023-02-28 17:00:00  0.003725

[43276 rows x 2 columns]
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000752 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1923
[LightGBM] [Info] Number of data points in the train set: 27796, number of used features: 11
[LightGBM] [Warning] 

In [9]:
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [10]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))

In [11]:
from itertools import product

In [21]:
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        continue
    print(f'{system_id}-{name}')
    if val == 'None':
        real_val = None
    else:
        real_val = val
    print(basic_lightbgm_test(system_id, f'./test_results/{system_id}-{val}', real_val, systems_cleaned, params))

4-other
                     time    energy
0     2007-09-01 06:00:00  0.032494
1     2007-09-01 07:00:00  0.159633
2     2007-09-01 08:00:00  0.304443
3     2007-09-01 09:00:00  0.265836
4     2007-09-01 10:00:00  0.804245
...                   ...       ...
43271 2023-02-28 13:00:00  0.610406
43272 2023-02-28 14:00:00  0.408742
43273 2023-02-28 15:00:00  0.395145
43274 2023-02-28 16:00:00  0.230329
43275 2023-02-28 17:00:00  0.003725

[43276 rows x 2 columns]
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000678 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1923
[LightGBM] [Info] Number of data points in the train set: 27796, number of used features: 11
[LightGBM] [W

ValueError: Input data must be 2 dimensional and non empty.

In [12]:
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        continue
    print(f'{system_id}-{name}')
    if val == 'None':
        real_val = None
    else:
        real_val = val
    trial_data = PreRun(system_id, f'./test_results/{system_id}-{val}', real_val, systems_cleaned)
    trial_data.good_end_days_naive(streak=2)


4-other
           date  streak_id
0    2007-09-01          1
1    2007-09-03          2
2    2007-09-04          2
3    2007-09-07          3
4    2007-09-08          3
...         ...        ...
3593 2023-02-21        645
3594 2023-02-25        646
3595 2023-02-26        646
3596 2023-02-27        646
3597 2023-02-28        646

[3598 rows x 2 columns]
10-other
           date  streak_id
0    2006-01-25          1
1    2006-01-26          1
2    2006-01-27          1
3    2006-01-28          1
4    2006-01-29          1
...         ...        ...
4165 2023-02-21        951
4166 2023-02-25        952
4167 2023-02-26        952
4168 2023-02-27        952
4169 2023-02-28        952

[4170 rows x 2 columns]
33-other
           date  streak_id
0    2010-11-10          1
1    2010-11-12          2
2    2010-11-13          2
3    2010-11-14          2
4    2010-11-16          3
...         ...        ...
3589 2023-02-21        502
3590 2023-02-25        503
3591 2023-02-26        503
3592 2

In [15]:
shorter_test = PreRun(4903, f'./test_results/4903-inverter', 'inverter', systems_cleaned)
shorter_test.add_weather_features_only()

In [17]:
shorter_test.good_end_days_naive(streak=5)

           date  streak_id
0    2014-08-02          1
1    2014-08-03          1
2    2014-08-08          2
3    2014-08-09          2
4    2014-08-10          2
...         ...        ...
1034 2018-02-03        139
1035 2018-02-05        140
1036 2018-02-14        141
1037 2018-02-15        141
1038 2018-02-16        141

[1039 rows x 2 columns]


,date
0,2014-08-13
1,2014-09-16
2,2014-09-24
3,2014-10-14
4,2014-11-05
...,...
77,2017-11-18
78,2017-11-24
79,2017-12-08
80,2017-12-22
